In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── File ─────────────────────────────────────────────────────────────────
DATA_FILE = "/uufs/chpc.utah.edu/common/home/lin-group24/agm/Mobile_SLV/Data/2026_old/v1.0-etl-freeze/merged/20260209.csv"

# ── Resample ──────────────────────────────────────────────────────────────
RESAMPLE_FREQ = '1s'

# ── Unit conversions (applied after resample, before zero-shift) ──────────
# new_col: (source_col, scale_factor)
UNIT_CONVERSIONS = {
    'C2H6_ppb_Ultra321': ('C2H6_ppm_Ultra321', 1000),
}

# ── Zero-shift (for single-species plots) ─────────────────────────────────
# Shifts every plotted column so its Nth-percentile matches ZERO_REF's Nth-percentile.
# Set ZERO_REF = None to zero each column to its own Nth-percentile (baseline → 0).
ZERO_PERCENTILE = 5
ZERO_REF        = 'CH4_ppm_Picarro'   # C2H6: use 'C2H6_ppb_Ultra460'

# ── Colors ────────────────────────────────────────────────────────────────
COLORS = {
    'Picarro':  '#1f77b4',
    'Ultra460': '#ff7f0e',
    'Ultra321': '#2ca02c',
    'Pico017':  '#d62728',
    'Anem':     '#9467bd',
    'GPS':      '#8c564b',
}

# ── Single-species stacked panels + overlay ───────────────────────────────
# PANELS  : (col, display_name, y_axis_label, color)  — one trace per row, shared x
# OVERLAY : (col, display_name, color)                — all on one axis
PANELS = [
    ('CH4_ppm_Picarro',  'Picarro',  'CH4 (ppm)', COLORS['Picarro']),
    ('CH4_ppm_Ultra460', 'Ultra460', 'CH4 (ppm)', COLORS['Ultra460']),
    ('CH4_ppm_Ultra321', 'Ultra321', 'CH4 (ppm)', COLORS['Ultra321']),
]
OVERLAY = [
    ('CH4_ppm_Picarro',  'Picarro',  COLORS['Picarro']),
    ('CH4_ppm_Ultra460', 'Ultra460', COLORS['Ultra460']),
]

# C2H6 — swap in by replacing the blocks above
# ZERO_REF = 'C2H6_ppb_Ultra460'
# PANELS = [
#     ('C2H6_ppb_Ultra460', 'Ultra460', 'C2H6 (ppb)', COLORS['Ultra460']),
#     ('C2H6_ppb_Ultra321', 'Ultra321', 'C2H6 (ppb)', COLORS['Ultra321']),
# ]
# OVERLAY = [
#     ('C2H6_ppb_Ultra460', 'Ultra460', COLORS['Ultra460']),
#     ('C2H6_ppb_Ultra321', 'Ultra321', COLORS['Ultra321']),
# ]

# ── Multi-species stacked panels (multiple traces per row) ────────────────
# zero_ref: column whose Nth-percentile sets the baseline for every trace in
#           that panel. None = zero each trace to its own Nth-percentile.
#           The ref column does NOT need to appear in traces[].
MULTI_PANELS = [
    dict(ylabel='CH4 (ppm)', zero_ref='CH4_ppm_Ultra460', traces=[
        ('CH4_ppm_Ultra460', 'WYO Ultra460', COLORS['Ultra460']),
        ('CH4_ppm_Ultra321', 'LANL Ultra321', COLORS['Ultra321']),
        ('CH4_ppm_Pico017',  'LANL Pico017',  COLORS['Pico017']),
    ]),
    dict(ylabel='C2H6 (ppb)', zero_ref='C2H6_ppb_Ultra460', traces=[
        ('C2H6_ppb_Ultra460', 'WYO Ultra460', COLORS['Ultra460']),
        ('C2H6_ppb_Ultra321', 'LANL Ultra321', COLORS['Ultra321']),
        ('C2H6_ppb_Pico017',  'LANL Pico017',  COLORS['Pico017']),
    ]),
    dict(ylabel='C3H8 (ppm)', zero_ref=None, traces=[
        ('C3H8_ppm_Ultra321', 'Ultra321', COLORS['Ultra321']),
    ]),
]

# ── Plot style ────────────────────────────────────────────────────────────
PLOT_STYLE = {
    'grid_color':       '#eeeeee',
    'legend_font_size': 9,
    'line_width':       1.5,
    'fig_width':        1150,
    'panel_height':     220,
    'margin_right':     80,
}

In [ ]:
# ── Load + resample + convert + zero-shift ───────────────────────────────
df = pd.read_csv(DATA_FILE, parse_dates=['TIMESTAMP'], index_col='TIMESTAMP')
df = df.resample(RESAMPLE_FREQ).mean().interpolate(method='time')

for new_col, (src_col, factor) in UNIT_CONVERSIONS.items():
    if src_col in df.columns:
        df[new_col] = df[src_col] * factor

all_cols = {p[0] for p in PANELS} | {o[0] for o in OVERLAY}
ref_base = df[ZERO_REF].dropna().quantile(ZERO_PERCENTILE / 100) if ZERO_REF else 0.0
for col in all_cols:
    if col in df.columns:
        df[col] = df[col] + (ref_base - df[col].dropna().quantile(ZERO_PERCENTILE / 100))

print(df.shape)
df.head()

In [ ]:
# ── Stacked panels ────────────────────────────────────────────────────────
PS  = PLOT_STYLE
n   = len(PANELS)
fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.04)

for i, (col, name, ylabel, color) in enumerate(PANELS, start=1):
    fig.add_trace(go.Scatter(
        x=df.index, y=df[col],
        mode='lines',
        line=dict(color=color, width=PS['line_width']),
        name=name,
        hovertemplate='%{y:.4f}<extra>' + name + '</extra>',
    ), row=i, col=1)
    fig.update_yaxes(title_text=f'{name}<br>{ylabel}', row=i, col=1,
                     showgrid=True, gridcolor=PS['grid_color'])

fig.update_xaxes(title_text='Time (UTC)', row=n, col=1)
fig.update_layout(
    title=dict(text='20260203 — stacked panels', font=dict(size=13)),
    hovermode='x unified',
    template='plotly_white',
    showlegend=False,
    width=PS['fig_width'],
    height=PS['panel_height'] * n * .9,
    margin=dict(r=PS['margin_right']),
)
fig.show()

In [ ]:
# ── Overlay ───────────────────────────────────────────────────────────────
PS  = PLOT_STYLE
fig = go.Figure()

for col, name, color in OVERLAY:
    fig.add_trace(go.Scatter(
        x=df.index, y=df[col],
        mode='lines',
        line=dict(color=color, width=PS['line_width']),
        name=name,
        hovertemplate='%{y:.4f}<extra>' + name + '</extra>',
    ))

fig.update_layout(
    title=dict(text='20260203 — overlay', font=dict(size=13)),
    xaxis=dict(title='Time (UTC)'),
    yaxis=dict(showgrid=True, gridcolor=PS['grid_color']),
    hovermode='x unified',
    template='plotly_white',
    showlegend=False,
    width=PS['fig_width'],
    height=PS['panel_height'] * 2,
    margin=dict(r=PS['margin_right']),
)
fig.show()

In [ ]:
# ── Multi-species stacked panels ─────────────────────────────────────────
PS   = PLOT_STYLE
n    = len(MULTI_PANELS)
seen = set()
fig  = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.04)

for i, panel in enumerate(MULTI_PANELS, start=1):
    ref_col  = panel.get('zero_ref')
    ref_base = df[ref_col].dropna().quantile(ZERO_PERCENTILE / 100) if ref_col and ref_col in df.columns else None

    for col, name, color in panel['traces']:
        if col not in df.columns:
            continue
        base     = ref_base if ref_base is not None else df[col].dropna().quantile(ZERO_PERCENTILE / 100)
        zeroed   = df[col] + (base - df[col].dropna().quantile(ZERO_PERCENTILE / 100))
        first    = name not in seen
        seen.add(name)
        fig.add_trace(go.Scatter(
            x=df.index, y=zeroed,
            mode='lines',
            line=dict(color=color, width=PS['line_width']),
            name=name,
            legendgroup=name,
            showlegend=first,
            hovertemplate='%{y:.4f}<extra>' + name + '</extra>',
        ), row=i, col=1)

    fig.update_yaxes(title_text=panel['ylabel'], row=i, col=1,
                     showgrid=True, gridcolor=PS['grid_color'])

fig.update_xaxes(title_text='Time (UTC)', row=n, col=1)
fig.update_layout(
    title=dict(text='20260203 — multi-species', font=dict(size=13)),
    hovermode='x unified',
    template='plotly_white',
    showlegend=True,
    width=PS['fig_width'],
    height=PS['panel_height'] * n,
    margin=dict(r=PS['margin_right'] + 80),
    legend=dict(font=dict(size=PS['legend_font_size']), x=1.02, y=1, xanchor='left'),
)
fig.show()